# Curated BUSI — Pré-processamento

Este notebook executa, passo a passo, o mesmo pré-processamento de
`src/dataset/Curated_BUSI_preprocessing.py`. Ele **importa** `src.dataset.paths` e
`src.dataset.preprocessing_utils` em vez de redeclarar caminhos e helpers, para não divergir do
script.

## Checklist de pré-requisitos

Antes de executar, confirme:
- [ ] Ambiente virtual ativo (`source .venv/bin/activate`)
- [ ] Dependências instaladas (`uv pip install -r requirements.txt`)
- [ ] Dataset original em `data/Curated_BUSI/raw/` com subpastas `benign/`, `malignant/`, `normal/`
- [ ] Arquivo `data/Curated_BUSI/curation_list.csv` presente (necessário se `CURATED=True`)
- [ ] `src/config.yaml` com `data.dataset: Curated_BUSI` — a Célula 3 aborta se apontar para outro dataset

## Layout

Nenhum caminho é fixo aqui: todos saem de `data.root` / `data.dataset` / `data.variant` em
`src/config.yaml`, resolvidos por `src/dataset/paths.py`.

```
data/<dataset>/
    raw/          download original extraído   <- entrada deste notebook
    archives/     .zip originais
    <variant>/    images/, masks/, mapping.csv <- saída deste notebook
    federated/    federated_mapping.csv
```

## O que este notebook faz

1. Valida o ambiente
2. Define a raiz do projeto e importa os módulos do repositório
3. Lê o `config.yaml`, resolve os caminhos e aplica as travas de segurança
4. Valida estrutura de pastas e arquivos
5. Carrega dataframes por classe
6. Identifica imagens com múltiplas máscaras
7. Carrega os IDs da lista de curadoria
8. Cria diretórios de saída
9. Redimensiona e combina imagens/máscaras
10. Gera o `mapping.csv` com metadados
11. Exibe resumo final e próximos passos

## Célula 1 — Validação de ambiente

In [ ]:
import sys
import importlib

print(f"Python: {sys.version}")

required_packages = {
    'numpy': 'numpy',
    'pandas': 'pandas',
    'cv2': 'opencv-python',
    'yaml': 'pyyaml',
    'pathlib': 'pathlib (stdlib)',
}

all_ok = True
for module, pkg_name in required_packages.items():
    try:
        importlib.import_module(module)
        print(f"  [OK] {pkg_name}")
    except ImportError:
        print(f"  [MISSING] {pkg_name} — instale com: uv pip install {pkg_name}")
        all_ok = False

if not all_ok:
    raise EnvironmentError("Dependências faltando. Instale antes de continuar.")
else:
    print("\nAmbiente OK.")

## Célula 2 — Raiz do projeto e imports

`data.root` (`data`) é relativo à raiz do repositório, e é de lá que o script roda. Trocar o `cwd`
aqui faz o notebook resolver exatamente os mesmos caminhos que ele.

In [ ]:
import os
from pathlib import Path
from typing import List

import cv2
import numpy as np
import pandas as pd
import yaml


def find_project_root(start: Path) -> Path:
    """Raiz do repo = primeira pasta, subindo a partir de `start`, que contém src/config.yaml."""
    for candidate in [start, *start.parents]:
        if (candidate / "src" / "config.yaml").exists():
            return candidate
    raise FileNotFoundError(
        "src/config.yaml não encontrado. Execute este notebook de dentro do repositório."
    )


# Idempotente: reexecutar a célula, já na raiz, não muda nada.
PROJECT_ROOT = find_project_root(Path.cwd())
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.dataset import paths
from src.dataset.preprocessing_utils import (add_image_metadata, assert_target_dataset,
                                             assert_variant_resolution, resize_image)

print(f"Project root: {PROJECT_ROOT}")
print(f"numpy  {np.__version__}")
print(f"pandas {pd.__version__}")
print(f"cv2    {cv2.__version__}")

## Célula 3 — Configuração e resolução de caminhos

Espelha o topo de `src/dataset/Curated_BUSI_preprocessing.py`. Para mudar dataset, variant,
resolução ou classes, edite `src/config.yaml` — não este notebook.

In [ ]:
CONFIG_FILE = "./src/config.yaml"
DATASET = "Curated_BUSI"

# Curated BUSI foi construído com vizinho mais próximo. INTER_AREA seria a melhor escolha para as
# imagens, mas mudar agora alteraria o dataset curado e invalidaria a partição federada congelada
# e todos os resultados derivados dela.
INTERPOLATION = cv2.INTER_NEAREST

# True => mantém apenas as imagens listadas em curation_list.csv (duplicatas removidas via SSIM).
# Com False toda imagem raw é mantida — aponte `data.variant` para outra pasta antes, senão o
# variant curado é sobrescrito.
CURATED = True
CURATION_LIST_FILENAME = "curation_list.csv"

with open(CONFIG_FILE) as cf:
    config_data = yaml.load(cf, Loader=yaml.FullLoader)["data"]

# Mesmas travas do script: não escrever no dataset errado, não misturar resoluções num variant.
assert_target_dataset(config_data, DATASET)
assert_variant_resolution(paths.mapping_file(config_data), config_data["image_size"])

class_names = config_data["classes"]
image_size = config_data["image_size"]
input_path = paths.raw_dir(config_data)
output_path = paths.processed_dir(config_data)
curation_list_path = paths.dataset_dir(config_data) / CURATION_LIST_FILENAME

print(f"Dataset  : {config_data['dataset']}")
print(f"Variant  : {config_data['variant']}")
print(f"Input    : {input_path}")
print(f"Output   : {output_path}")
print(f"Curadoria: {curation_list_path}")
print(f"Classes  : {class_names}")
print(f"Curated  : {CURATED}")
print(f"Resize   : ({image_size}, {image_size})")

## Célula 4 — Validação de estrutura de pastas e arquivos

In [ ]:
import shutil

class_subfolders_missing = [
    cls for cls in class_names
    if not (input_path / cls).exists()
]

if not class_subfolders_missing:
    print("Estrutura de classes já existe. Nenhuma ação necessária.")
else:
    print(f"Subpastas ausentes: {class_subfolders_missing}")

    # Recuperação: havendo apenas imagens já pré-processadas em raw/images, reconstrói as pastas de
    # classe com a nomenclatura original do BUSI, que é o que o resto do notebook espera.
    processed_images_dir = input_path / "images"
    processed_masks_dir = input_path / "masks"

    if not processed_images_dir.exists() or not list(processed_images_dir.glob("*.png")):
        print("\n[ERRO] Nenhum dado encontrado em images/ para recuperar a estrutura.")
        print("Baixe o dataset BUSI original em: https://scholar.cu.edu.eg/?q=afahmy/pages/dataset")
        print("E extraia de forma que a estrutura seja:")
        for cls in class_names:
            print(f"  {input_path / cls}/")
        raise FileNotFoundError("Dataset BUSI original não encontrado.")

    # benign_id_100.png      → benign/benign (100).png
    # benign_id_100_mask.png → benign/benign (100)_mask.png
    print(f"\nEncontrados arquivos pré-processados em '{processed_images_dir}'. Recriando estrutura...")

    for cls in class_names:
        (input_path / cls).mkdir(exist_ok=True)

    img_count = {cls: 0 for cls in class_names}
    skip_count = 0

    for img_file in sorted(processed_images_dir.glob("*.png")):
        parts = img_file.stem.split("_id_")   # ["benign", "100"]
        if len(parts) != 2 or parts[0] not in class_names:
            skip_count += 1
            continue
        cls, id_str = parts
        dest_img = input_path / cls / f"{cls} ({id_str}).png"
        dest_mask = input_path / cls / f"{cls} ({id_str})_mask.png"
        shutil.copy2(str(img_file), str(dest_img))

        mask_file = processed_masks_dir / f"{cls}_id_{id_str}_mask.png"
        if mask_file.exists():
            shutil.copy2(str(mask_file), str(dest_mask))

        img_count[cls] += 1

    print("\nEstrutura recriada com sucesso:")
    for cls in class_names:
        print(f"  {cls:12s} — {img_count[cls]} imagens copiadas → {input_path / cls}")
    if skip_count:
        print(f"  ({skip_count} arquivo(s) ignorado(s) por não seguir o padrão {{classe}}_id_{{n}}.png)")

In [ ]:
errors = []

# 1. Pasta raw
if not input_path.exists():
    errors.append(f"[MISSING] Pasta raw não encontrada: {input_path}\n"
                  "  -> Baixe o dataset BUSI em https://scholar.cu.edu.eg/?q=afahmy/pages/dataset")
else:
    print(f"[OK] Pasta raw: {input_path}")
    for cls in class_names:
        cls_path = input_path / cls
        if not cls_path.exists():
            errors.append(f"[MISSING] Subpasta de classe ausente: {cls_path}")
        else:
            pngs = list(cls_path.glob("*.png"))
            print(f"  [OK] {cls:12s} — {len(pngs)} arquivos .png")
            if len(pngs) == 0:
                errors.append(f"[EMPTY] {cls_path} não contém arquivos .png")

# 2. Lista de curadoria
if CURATED:
    if not curation_list_path.exists():
        errors.append(f"[MISSING] Lista de curadoria: {curation_list_path}")
    else:
        curated_df = pd.read_csv(curation_list_path, sep=';')
        required_cols = {'class', 'id'}
        missing_cols = required_cols - set(curated_df.columns)
        if missing_cols:
            errors.append(f"[INVALID] curation_list.csv sem colunas obrigatórias: {missing_cols}")
        else:
            print(f"[OK] Lista de curadoria: {len(curated_df)} linhas, colunas: {list(curated_df.columns)}")

if errors:
    print("\n=== ERROS ENCONTRADOS ===")
    for e in errors:
        print(e)
    raise FileNotFoundError("Corrija os erros acima antes de continuar.")
else:
    print("\nEstrutura de dados validada com sucesso.")

## Célula 5 — Carregar dataframes por classe

In [ ]:
def load_class_dataframe(class_path: Path, class_name: str) -> pd.DataFrame:
    files = [f for f in sorted(os.listdir(class_path)) if f.endswith(".png")]
    ids = [
        f.replace(".png", "").split(" ")[-1].split("_")[0]
        .replace("(", "").replace(")", "")
        for f in files
    ]
    types = ["mask" if "mask" in f else "img" for f in files]
    df = pd.DataFrame({"class": [class_name] * len(ids), "ids": ids, "type": types})
    print(f"  {class_name:12s} — {len(df[df['type']=='img'])} imagens, {len(df[df['type']=='mask'])} máscaras")
    return df

print("Carregando dataframes por classe:")
df_classes = [load_class_dataframe(input_path / cls, cls) for cls in class_names]

# Visualização rápida
for df, cls in zip(df_classes, class_names):
    display(df.head(5))

## Célula 6 — Identificar imagens com múltiplas máscaras

In [ ]:
def get_mask_counts(df: pd.DataFrame, class_name: str) -> List[int]:
    counts = df.groupby("ids").apply(lambda x: sum(x['type'] == 'mask')).to_dict()
    multi = [int(k) for k, v in counts.items() if v > 1]
    print(f"  {class_name:12s} — {len(multi)} imagens com múltiplas máscaras")
    return multi

print("Imagens com múltiplas máscaras (serão combinadas):")
multi_masks_per_class = [
    get_mask_counts(df, cls)
    for df, cls in zip(df_classes, class_names)
]

## Célula 7 — Carregar IDs da lista de curadoria

In [ ]:
if CURATED:
    curated_mapping = pd.read_csv(curation_list_path, sep=';')
    curated_ids_dict = {
        cls: curated_mapping[curated_mapping['class'] == cls]['id'].astype(int).tolist()
        for cls in class_names
    }
    for cls, ids in curated_ids_dict.items():
        print(f"  {cls:12s} — {len(ids)} IDs na lista de curadoria")
else:
    curated_ids_dict = {cls: None for cls in class_names}
    print("Modo não-curado: todas as imagens serão processadas.")

## Célula 8 — Criar diretórios de saída

In [ ]:
assert input_path.exists(), f"Raw dataset folder '{input_path}' does not exist"
(output_path / "images").mkdir(parents=True, exist_ok=True)
(output_path / "masks").mkdir(parents=True, exist_ok=True)

print(f"[OK] {output_path / 'images'}")
print(f"[OK] {output_path / 'masks'}")

## Célula 9 — Redimensionar e salvar imagens/máscaras

In [ ]:
def combine_and_resize_images(
    class_name: str,
    class_ids: List[str],
    multi_mask_ids: List[int],
    path: Path,
    out_path: Path,
    image_size: int,
    curated_ids: List[int] = None
) -> int:
    saved = 0
    for j in set(class_ids):
        j_int = int(j)
        if curated_ids is not None and j_int not in curated_ids:
            continue

        img_file = path / class_name / f"{class_name} ({j}).png"
        if not img_file.exists():
            print(f"  [WARN] Imagem não encontrada: {img_file}")
            continue

        img = cv2.imread(str(img_file), 0)

        mask_files = [f"{class_name} ({j})_mask.png"]
        if j_int in multi_mask_ids:
            mask_files.append(f"{class_name} ({j})_mask_1.png")

        total_mask = sum(
            cv2.imread(str(path / class_name / mf), 0)
            for mf in mask_files
            if (path / class_name / mf).exists()
        )

        img_resized = resize_image(img, image_size, INTERPOLATION)
        mask_resized = resize_image(total_mask, image_size, INTERPOLATION)

        cv2.imwrite(str(out_path / "images" / f"{class_name}_id_{j}.png"), img_resized)
        cv2.imwrite(str(out_path / "masks" / f"{class_name}_id_{j}_mask.png"), mask_resized)
        saved += 1

    return saved

print("Processando imagens e máscaras:")
for i, (df, cls) in enumerate(zip(df_classes, class_names)):
    n = combine_and_resize_images(
        cls,
        df['ids'].tolist(),
        multi_masks_per_class[i],
        input_path,
        output_path,
        image_size,
        curated_ids_dict.get(cls)
    )
    print(f"  {cls:12s} — {n} imagens salvas")

print("\nProcessamento concluído.")

## Célula 10 — Gerar o mapping.csv

In [ ]:
img_paths = sorted((output_path / "images").glob("*.png"))

# O caminho da máscara vem do stem da imagem, e não de substituir "images" -> "masks" na string:
# a própria pasta do dataset pode conter qualquer uma das duas palavras.
mask_paths = [output_path / "masks" / f"{p.stem}_mask.png" for p in img_paths]
df_mapping = pd.DataFrame({"img_path": [p.as_posix() for p in img_paths],
                           "mask_path": [p.as_posix() for p in mask_paths]})
df_mapping['class'] = df_mapping['img_path'].apply(lambda x: Path(x).stem.split('_')[0])
df_mapping['id'] = df_mapping['img_path'].apply(lambda x: int(Path(x).stem.split('_')[-1]))

print(f"Total de imagens mapeadas: {len(df_mapping)}")
display(df_mapping.head())

## Célula 11 — Adicionar metadados (dimensões, pixels de tumor, bounding box)

`add_image_metadata` relê do disco o que foi escrito, então os números descrevem a saída real.
As colunas de máscara ficam `NaN` nas linhas sem máscara — `0` significaria máscara vazia, que é
legítimo para a classe `normal`.

In [ ]:
df_mapping = add_image_metadata(df_mapping)
display(df_mapping.head())

## Célula 12 — Salvar CSV e exibir resumo

In [ ]:
mapping_csv_path = paths.mapping_file(config_data)
df_mapping.to_csv(str(mapping_csv_path), index=False)

print(f"[OK] Mapeamento salvo em: {mapping_csv_path}")
print(f"\nTotal de imagens processadas: {len(df_mapping)}")
for cls in class_names:
    n = len(df_mapping[df_mapping['class'] == cls])
    print(f"  {cls.capitalize():12s}: {n} imagens")

## Próximos passos

O pré-processamento foi concluído: `data/Curated_BUSI/processed_128/` agora contém `images/`,
`masks/` e `mapping.csv`.

1. **Nada a editar em `src/config.yaml`** — o treinamento lê o que acabou de ser gerado a partir
   das mesmas chaves que este notebook usou:
   ```yaml
   data:
     root: data
     dataset: Curated_BUSI
     variant: processed_128
     image_size: 128
   ```
   Os caminhos são derivados dessas chaves por `src/dataset/paths.py`; não há caminho fixo para
   ajustar.

2. **Gere a partição federada mestre** (uma vez só, e depois congelada — é o que faz a comparação
   entre os setups ser justa):
   ```bash
   python -m src.dataset.federated_partition
   ```

3. **Execute os treinamentos**:
   ```bash
   python -m src.training_federated      # federated (federated.standalone: False)
   # editar config -> federated.standalone: True, e rodar de novo: standalone (local-only)
   python -m src.training_centralized    # centralized MTL
   python -m src.experiments.analyze     # análise comparativa
   ```